<a href="https://colab.research.google.com/github/heyanugrah/Transformers/blob/main/Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Basic Transformer- Encoder

In [1]:
import torch
import torch.nn as nn
import math

In [2]:
# 1. Define the Vocabulary (Adding '[sep]')
new_vocab = {'<pad>': 0, '<unk>': 1, '<cls>': 2, '[sep]': 3, 'this': 4, 'is': 5, 'a': 6, 'positive': 7, 'sentence': 8, 'learning': 9, 'about': 10, 'transformers': 11, 'fun': 12, 'i': 13, 'hate': 14, 'am': 15, 'so': 16, 'disappointed': 17, 'really': 18, 'enjoyed': 19, 'movie': 20, 'wonderful': 21, 'day': 22, 'for': 23, 'walk': 24, 'product': 25, 'excellent!': 26, 'absolutely': 27, 'dreadful': 28, 'service': 29, 'what': 30, 'terrible': 31, 'experience': 32, 'feeling': 33, 'very': 34, 'happy': 35, 'today': 36, 'boy': 37, 'anugrah': 38}

# 2. Define the Reverse Vocabulary
idx_to_word = {i: word for word, i in new_vocab.items()}

vocab_size = len(new_vocab)

print(f"Vocabulary size is: {vocab_size}")

Vocabulary size is: 39


### Modular Tokenization Functions

To improve clarity and reusability, let's break down the tokenization process into several distinct functions:

1.  `_add_special_tokens_and_segments`: Adds `<cls>` and `[sep]` tokens and initializes segment IDs.
2.  `_convert_to_ids`: Converts token strings to their numerical IDs.
3.  `_pad_ids_and_segments`: Handles padding for both token IDs and segment IDs.
4.  `_create_attention_mask_from_ids`: Generates the attention mask from padded token IDs.
5.  `prepare_transformer_inputs`: An orchestrating function that calls the above helpers to produce all necessary inputs for a Transformer model.

In [3]:
# Helper to add special tokens and initial segment IDs
def _add_special_tokens_and_segments(input_text, vocab):
    tokens = [vocab.get("<cls>", "<cls>")] # Start with <cls>
    segment_ids = [0] # <cls> belongs to segment 0

    # Tokenize input text and add words
    for word in input_text.lower().replace('.', '').split():
        tokens.append(word)
        segment_ids.append(0) # All words in the first sentence belong to segment 0

    tokens.append(vocab.get("[sep]", "[sep]")) # Add [sep]
    segment_ids.append(0) # [sep] belongs to segment 0

    return tokens, segment_ids

# Helper to convert tokens to IDs
def _convert_to_ids(tokens, vocab):
    return [vocab.get(token, vocab["<unk>"]) for token in tokens]

# Helper to pad token IDs and segment IDs
def _pad_ids_and_segments(token_ids, segment_ids, max_seq_len, pad_token_id):
    if len(token_ids) > max_seq_len:
        token_ids = token_ids[:max_seq_len]
        segment_ids = segment_ids[:max_seq_len]
    else:
        padding_length = max_seq_len - len(token_ids)
        token_ids.extend([pad_token_id] * padding_length)
        segment_ids.extend([0] * padding_length) # Padding tokens belong to segment 0

    return token_ids, segment_ids

# Helper to create attention mask
def _create_attention_mask_from_ids(token_ids, pad_token_id):
    mask = [1 if token_id != pad_token_id else 0 for token_id in token_ids]
    return torch.tensor(mask, dtype=torch.long).unsqueeze(0).unsqueeze(1).unsqueeze(2) # [batch_size, 1, 1, seq_len]

In [4]:
def prepare_transformer_inputs(input_text, vocab, max_seq_len):
    pad_token_id = vocab['<pad>']

    # Step 1: Add special tokens and assign initial segment IDs
    tokens, segment_ids = _add_special_tokens_and_segments(input_text, vocab)

    # Step 2: Convert tokens to IDs
    token_ids = _convert_to_ids(tokens, vocab)

    # Step 3: Pad token IDs and segment IDs
    padded_token_ids, padded_segment_ids = _pad_ids_and_segments(
        token_ids, segment_ids, max_seq_len, pad_token_id
    )

    # Convert to PyTorch tensors and add batch dimension
    input_ids = torch.tensor(padded_token_ids, dtype=torch.long).unsqueeze(0)
    token_type_ids = torch.tensor(padded_segment_ids, dtype=torch.long).unsqueeze(0)

    # Step 4: Create attention mask
    attention_mask = _create_attention_mask_from_ids(padded_token_ids, pad_token_id)

    return input_ids, attention_mask, token_type_ids

### Demonstration of the new tokenization pipeline

In [5]:


input_text = "anugrah is learning"
max_seq_len = 10 # maximum token need to be created

# Prepare inputs using the new modular function
input_ids, attention_mask_new, token_type_ids_new = prepare_transformer_inputs(
    input_text, new_vocab, max_seq_len
)

print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids}")
print(f"Attention Mask: {attention_mask_new}")
print(f"Token Type IDs: {token_type_ids_new}")

print(f"Input IDs shape: {input_ids.shape}")
print(f"Attention Mask shape: {attention_mask_new.shape}")
print(f"Token Type IDs shape: {token_type_ids_new.shape}")

Input Text: 'anugrah is learning'
Input IDs: tensor([[ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0]])
Attention Mask: tensor([[[[1, 1, 1, 1, 1, 0, 0, 0, 0, 0]]]])
Token Type IDs: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
Input IDs shape: torch.Size([1, 10])
Attention Mask shape: torch.Size([1, 1, 1, 10])
Token Type IDs shape: torch.Size([1, 10])


### Create and apply embedding

In [6]:
d_model = 8 #vector dimension for each word

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=d_model
)

embedded_input = embedding(input_ids)
print(f"Shape of embedded input: {embedded_input.shape}")

Shape of embedded input: torch.Size([1, 10, 8])


In [7]:
print(embedded_input)

tensor([[[ 0.7989,  0.7531, -1.3184,  0.9312,  0.6105,  0.6229,  1.5397,
          -0.7760],
         [ 1.5964, -0.8608, -1.2149,  0.8729, -0.0764,  0.7593, -1.2578,
          -0.4609],
         [-0.1346, -0.5892, -0.1714,  0.5189,  0.9471, -0.3500,  0.4711,
          -0.1291],
         [ 0.1428,  0.1146,  0.7287, -1.3843,  0.2919, -1.6149, -0.3424,
           1.5984],
         [ 0.7989,  0.7531, -1.3184,  0.9312,  0.6105,  0.6229,  1.5397,
          -0.7760],
         [-0.3492,  1.0742,  0.3288,  0.5611, -0.2295,  2.1748,  0.3439,
          -1.1819],
         [-0.3492,  1.0742,  0.3288,  0.5611, -0.2295,  2.1748,  0.3439,
          -1.1819],
         [-0.3492,  1.0742,  0.3288,  0.5611, -0.2295,  2.1748,  0.3439,
          -1.1819],
         [-0.3492,  1.0742,  0.3288,  0.5611, -0.2295,  2.1748,  0.3439,
          -1.1819],
         [-0.3492,  1.0742,  0.3288,  0.5611, -0.2295,  2.1748,  0.3439,
          -1.1819]]], grad_fn=<EmbeddingBackward0>)


In [8]:
print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids.squeeze(0)}")

print("\nEncodings for each token in 'anugrah is learning':")
# Iterate through the input_ids and corresponding embedded vectors
for i, token_id in enumerate(input_ids.squeeze(0)):
    word = idx_to_word.get(token_id.item(), '<unknown>')
    embedding_vector = embedded_input.squeeze(0)[i]
    print(f"Token: '{word}' (ID: {token_id.item()}) -> Embedding: {embedding_vector.tolist()}")

Input Text: 'anugrah is learning'
Input IDs: tensor([ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0])

Encodings for each token in 'anugrah is learning':
Token: '<unk>' (ID: 1) -> Embedding: [0.7988660931587219, 0.753094494342804, -1.318389654159546, 0.9311954379081726, 0.6105321049690247, 0.6228905916213989, 1.539669156074524, -0.7760450839996338]
Token: 'anugrah' (ID: 38) -> Embedding: [1.5963551998138428, -0.860789954662323, -1.214906930923462, 0.8729375004768372, -0.07643826305866241, 0.7592783570289612, -1.2577680349349976, -0.460890531539917]
Token: 'is' (ID: 5) -> Embedding: [-0.1346445232629776, -0.5892316699028015, -0.171426460146904, 0.5188692808151245, 0.9471042156219482, -0.34996116161346436, 0.4711131155490875, -0.1290895938873291]
Token: 'learning' (ID: 9) -> Embedding: [0.1427946537733078, 0.11459445208311081, 0.7286683917045593, -1.3842992782592773, 0.2918582856655121, -1.6148828268051147, -0.34241077303886414, 1.5983554124832153]
Token: '<unk>' (ID: 1) -> Embedding: [0.7988660

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=5000):
        super().__init__()

        self.dropout = nn.Dropout(0.1)

        # Create a positional encoding matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term) if d_model % 2 == 0 else torch.cos(position * div_term[:-1])

        pe = pe.unsqueeze(0) # Add batch dimension
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Add positional encoding to the input embeddings
        # x has shape (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# Instantiate Positional Encoding
pos_encoder = PositionalEncoding(d_model, max_seq_len)

# Apply positional encoding to the embedded input
embedded_with_pos = pos_encoder(embedded_input)

print(f"Shape of embedded input with positional encoding: {embedded_with_pos.shape}")

Shape of embedded input with positional encoding: torch.Size([1, 10, 8])


In [10]:
print(embedded_with_pos[0][1],embedded_with_pos.shape) #PE for anugrah (1,batch,10 total , 8 dimension)

tensor([ 2.7087, -0.0000, -1.2390,  0.0000, -0.0738,  1.9547, -1.3964,  0.5990],
       grad_fn=<SelectBackward0>) torch.Size([1, 10, 8])


In [11]:
print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids.squeeze(0)}")

print("\nCombined Embeddings (Word Embedding + Positional Encoding) for each token:")
for i, token_id in enumerate(input_ids.squeeze(0)):
    word = idx_to_word.get(token_id.item(), '<unknown>')
    combined_embedding_vector = embedded_with_pos.squeeze(0)[i]
    print(f"Token: '{word}' (ID: {token_id.item()}) -> Combined Embedding: {combined_embedding_vector.tolist()}")

Input Text: 'anugrah is learning'
Input IDs: tensor([ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0])

Combined Embeddings (Word Embedding + Positional Encoding) for each token:
Token: '<unk>' (ID: 1) -> Combined Embedding: [0.8876290321350098, 1.9478827714920044, -1.4648774862289429, 2.145772933959961, 0.0, 1.8032118082046509, 0.0, 0.24883881211280823]
Token: 'anugrah' (ID: 38) -> Combined Embedding: [2.708695888519287, -0.0, -1.2389706373214722, 0.0, -0.07382047921419144, 1.9546982049942017, -1.3964089155197144, 0.5990099906921387]
Token: 'is' (ID: 5) -> Combined Embedding: [0.860725462436676, -1.1170872449874878, 0.030269838869571686, 1.6654844284057617, 1.0745588541030884, 0.7220432162284851, 0.5256812572479248, 0.9676761031150818]
Token: 'learning' (ID: 9) -> Combined Embedding: [0.315460741519928, -0.9726645350456238, 1.1379872560501099, -0.47662532329559326, 0.35761532187461853, -0.6837031245231628, -0.3771231174468994, 2.887056827545166]
Token: '<unk>' (ID: 1) -> Combined Embedding: [0

In [12]:
token_type_embedding = nn.Embedding(2, d_model) # Assuming 2 types: 0 for sentence A, 1 for sentence B

'''
Segment Embedding

Tells the model which sentence a token belongs to.

Example:

Sentence A: I love AI
Sentence B: It is powerful

BERT input:

[CLS] I love AI [SEP] It is powerful [SEP]
'''

# Get segment embeddings for our token_type_ids
segment_embeddings = token_type_embedding(token_type_ids_new)

# Add the segment embeddings to the combined word and positional embeddings
final_transformer_input = embedded_with_pos + segment_embeddings

print(f"Shape of token_type_ids: {token_type_ids_new.shape}")
print(f"Shape of segment embeddings: {segment_embeddings.shape}")
print(f"Shape of final input to Transformer: {final_transformer_input.shape}")

print("\nSegment Embeddings:")
print(segment_embeddings)

Shape of token_type_ids: torch.Size([1, 10])
Shape of segment embeddings: torch.Size([1, 10, 8])
Shape of final input to Transformer: torch.Size([1, 10, 8])

Segment Embeddings:
tensor([[[ 0.3763,  1.5218, -0.5388,  0.3053, -0.2910,  1.4740,  0.0204,
           0.3650],
         [ 0.3763,  1.5218, -0.5388,  0.3053, -0.2910,  1.4740,  0.0204,
           0.3650],
         [ 0.3763,  1.5218, -0.5388,  0.3053, -0.2910,  1.4740,  0.0204,
           0.3650],
         [ 0.3763,  1.5218, -0.5388,  0.3053, -0.2910,  1.4740,  0.0204,
           0.3650],
         [ 0.3763,  1.5218, -0.5388,  0.3053, -0.2910,  1.4740,  0.0204,
           0.3650],
         [ 0.3763,  1.5218, -0.5388,  0.3053, -0.2910,  1.4740,  0.0204,
           0.3650],
         [ 0.3763,  1.5218, -0.5388,  0.3053, -0.2910,  1.4740,  0.0204,
           0.3650],
         [ 0.3763,  1.5218, -0.5388,  0.3053, -0.2910,  1.4740,  0.0204,
           0.3650],
         [ 0.3763,  1.5218, -0.5388,  0.3053, -0.2910,  1.4740,  0.0204,
     

In [13]:
print('final transformer input')
print(final_transformer_input)

final transformer input
tensor([[[ 1.2640,  3.4697, -2.0036,  2.4511, -0.2910,  3.2773,  0.0204,
           0.6139],
         [ 3.0850,  1.5218, -1.7777,  0.3053, -0.3648,  3.4287, -1.3760,
           0.9640],
         [ 1.2371,  0.4047, -0.5085,  1.9708,  0.7836,  2.1961,  0.5461,
           1.3327],
         [ 0.6918,  0.5492,  0.5992, -0.1713,  0.0666,  0.7903, -0.3567,
           3.2521],
         [ 0.4231,  1.6323, -1.5709,  2.3634,  0.4318,  3.2764,  1.7356,
           0.6139],
         [-1.0772,  3.0305,  0.3592,  1.9039, -0.4905,  5.0002,  0.4080,
           0.1630],
         [-0.3222,  3.7822,  0.4539,  1.8458, -0.4794,  4.9996,  0.4091,
           0.3650],
         [ 0.7183,  3.5530,  0.5424,  1.7786, -0.4683,  4.9989,  0.4102,
           0.1629],
         [ 1.0876,  2.5537,  0.6236,  1.7029, -0.4572,  4.9980,  0.4114,
           0.1629],
         [ 0.4462,  1.7030,  0.6969,  1.6195, -0.2910,  4.9971,  0.4125,
           0.1629]]], grad_fn=<AddBackward0>)


In [14]:
import torch
import torch.nn as nn
import math

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ffn_hidden_dim, dropout_rate=0.1):
        super().__init__()

        '''
          d_model = 512
          num_heads = 8

          head_dim = 64
          512 dimensions
          ├─ Head 1 : 64
          ├─ Head 2 : 64
          ├─ Head 3 : 64
          ├─ Head 4 : 64
          ├─ Head 5 : 64
          ├─ Head 6 : 64
          ├─ Head 7 : 64
          └─ Head 8 : 64
        '''

        assert d_model % num_heads == 0 #if it is divisible then return true and allow run

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Multi-Head Attention
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

        self.attn_dropout = nn.Dropout(dropout_rate)

        # LayerNorm after MHA residual
        self.layer_norm1 = nn.LayerNorm(d_model)

        # Feed Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, d_model)
        )

        self.ffn_dropout = nn.Dropout(dropout_rate)

        # LayerNorm after FFN residual
        self.layer_norm2 = nn.LayerNorm(d_model)

    def _calculate_attention(self, Q, K, V, mask=None):

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        ) / math.sqrt(self.head_dim) # The division by sqrt(head_dim) is the scaling factor.

        if mask is not None:
            # Apply padding mask to the scores. Positions where mask == 0 are set to -inf.
            # This prevents attention to padded tokens in the input sequence.
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        output = torch.matmul(
            attention_weights,
            V
        )

        return output

    def forward(self, x, mask=None):
        # In your current setup (cell cbe1d5c1), `mask` is indeed passed as a tensor
        # (attention_mask_new), not None, when calling this layer.

        batch_size, seq_len, _ = x.shape

        # ----------------------------
        # Multi-Head Self Attention
        # ----------------------------

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        '''
        Full token embedding (8 dims)
                  │
                  ▼
            Split into 2 heads
                  │
          ┌──────┴──────┐
          ▼             ▼
        Head 1        Head 2
        (4 dims)      (4 dims)
          │             │
        Attention    Attention
        separately   separately
        '''

        '''
        # Split Q into multiple attention heads and rearrange dimensions
        # From: [batch_size, seq_len, d_model]
        # To:   [batch_size, num_heads, seq_len, head_dim]
        '''
        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        attention_output = self._calculate_attention(
            Q,
            K,
            V,
            mask=mask
        )
        '''
        Before attention:

        Token
        [1,2,3,4,5,6,7,8]

        ↓ split

        Head1 [1,2,3,4]
        Head2 [5,6,7,8]

        After attention:
        Head1 [1,2,3,4]
        Head2 [5,6,7,8]

        ↓ concatenate
        combined attention heads : [1,2,3,4,5,6,7,8]
        '''
        attention_output = (
            attention_output
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                seq_len,
                self.d_model
            )
        )
        '''
        Multi-head outputs
              ↓
        Concatenate
              ↓
        Wo (Linear Layer)
              ↓
        Final attention representation

        [1,2,3,4,5,6,7,8]
            ↓
        Linear Wo
            ↓
       [5,-2,8,1,4,9,3,7]
        '''

        mha_output = self.Wo(
            attention_output
        )
        '''
        Some output values are randomly set to 0.Prevent Overfitting

        Before:
        [5,-2,8,1,4,9,3,7]

        After dropout:
        [5,0,8,1,0,9,3,7]
        '''
        final_mha_output = self.attn_dropout(
            mha_output
        )

        '''
          Input
            ↓
          Wq, Wk, Wv
            ↓
          Split into heads
            ↓
          Attention
            ↓
          Merge heads
            ↓
          Wo
            ↓
          Dropout
            ↓
          Final MHA output

        '''

        # Residual Connection + LayerNorm
        x = self.layer_norm1(
            x + final_mha_output
        )

        # ----------------------------
        # Feed Forward Network
        # ----------------------------

        ffn_output = self.ffn(x)

        ffn_output = self.ffn_dropout(
            ffn_output
        )

        # Residual Connection + LayerNorm
        output = self.layer_norm2(
            x + ffn_output
        )

        return output

# # --- Execution Code ---
# encoder_layer = TransformerEncoderLayer(d_model=8, num_heads=2, ffn_hidden_dim=32)
# encoder_output = encoder_layer(final_transformer_input, mask=attention_mask_new)

# print(f"Encoder Layer Input Shape: {final_transformer_input.shape}")
# print(f"Encoder Layer Output Shape: {encoder_output.shape}")
# print("\nFirst few values of the output:")
# print(encoder_output[0, 0, :])

In [17]:
'''
Input Target Sentence
        ↓
Embedding
        ↓
Positional Encoding
        ↓
Masked Self Attention
(Q,K,V from Decoder Input)
        ↓
Add + Norm
        ↓
Cross Attention
(Q from Decoder)

(K,V from Encoder Output)
        ↓
Add + Norm
        ↓
Feed Forward Network
        ↓
Add + Norm
        ↓
Decoder Output
'''

class TransformerDecoderLayer(nn.Module):

    def __init__(self,
                 d_model,
                 num_heads,
                 ffn_hidden_dim,
                 dropout_rate=0.1):

        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # ===============================
        # 1. Masked Self Attention
        # ===============================

        self.self_Wq = nn.Linear(d_model, d_model)
        self.self_Wk = nn.Linear(d_model, d_model)
        self.self_Wv = nn.Linear(d_model, d_model)

        self.self_Wo = nn.Linear(d_model, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout_rate)

        # ===============================
        # 2. Encoder Decoder Attention
        # ===============================

        self.cross_Wq = nn.Linear(d_model, d_model)

        self.cross_Wk = nn.Linear(d_model, d_model)
        self.cross_Wv = nn.Linear(d_model, d_model)

        self.cross_Wo = nn.Linear(d_model, d_model)

        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout_rate)

        # ===============================
        # 3. Feed Forward Network
        # ===============================

        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, d_model)
        )

        self.norm3 = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(dropout_rate)

    """
    Splits the embedding dimension (d_model) into multiple attention heads.

    Input Shape:
        (batch_size, seq_len, d_model)

    Example:
        (32, 10, 512)

    After Reshape:
        (batch_size, seq_len, num_heads, head_dim)

    Example:
        (32, 10, 8, 64)

    After Transpose:
        (batch_size, num_heads, seq_len, head_dim)

    Example:
        (32, 8, 10, 64)

    Purpose:
        Allows Multi-Head Attention to process multiple attention heads
        in parallel, where each head learns different relationships
        between tokens.
    """
    def split_heads(self, x):


        batch_size = x.shape[0]
        seq_len = x.shape[1]

        x = x.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        return x.transpose(1, 2)

    """
    Computes Scaled Dot-Product Attention.

    Steps:
    1. Calculate attention scores between Query (Q) and Key (K).
       Higher scores indicate stronger relationships between tokens.

    2. Scale the scores by sqrt(head_dim) to prevent very large values
       that can make softmax unstable.

    3. Apply the mask (if provided):
       - src_mask: ignores padding tokens.
       - tgt_mask: ignores padding tokens and future tokens.

    4. Apply Softmax to convert scores into attention weights
       (probabilities that sum to 1).

    5. Multiply attention weights by Value (V) to produce the
       final attention output.

    Input Shapes:
        Q : (batch_size, num_heads, seq_len_q, head_dim)
        K : (batch_size, num_heads, seq_len_k, head_dim)
        V : (batch_size, num_heads, seq_len_k, head_dim)

    Output Shape:
        (batch_size, num_heads, seq_len_q, head_dim)

    Formula:
        Attention(Q,K,V) =
        softmax((Q × Kᵀ) / √head_dim) × V

    Args:
        Q    : Query vectors.
        K    : Key vectors.
        V    : Value vectors.
        mask : Optional attention mask. If None, attention is
               computed normally without masking.
    """
    def attention(self, Q, K, V, mask=None):

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        scores = scores / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(
                mask == 0,
                float("-inf")
            )

        weights = torch.softmax(scores, dim=-1)

        output = torch.matmul(weights, V)

        return output

    """
    Performs one Transformer Decoder Layer forward pass.

    Args:
        decoder_input:
            Target sequence representation from the previous decoder
            layer (or embedding layer for the first decoder layer).

        encoder_output:
            Output produced by the encoder. Used by the decoder
            to access information from the source sequence.

        src_mask:
            Source padding mask used during encoder-decoder attention.
            Prevents the decoder from attending to PAD tokens in the
            encoder output.

        tgt_mask:
            Target mask used during masked self-attention.
            Combines:
            - Look-ahead (causal) mask to hide future tokens.
            - Padding mask to ignore PAD tokens.

    Processing Steps:

    Step 1: Masked Multi-Head Self-Attention
        - Generate Q, K, and V from decoder_input.
        - Apply tgt_mask to prevent attention to future words.
        - Combine attention heads.
        - Apply output projection (Wo).
        - Apply Residual Connection and LayerNorm.

    Step 2: Encoder-Decoder Multi-Head Attention
        - Q comes from decoder output.
        - K and V come from encoder output.
        - Apply src_mask to ignore encoder PAD tokens.
        - Allows the decoder to attend to relevant source tokens.
        - Apply Residual Connection and LayerNorm.

    Step 3: Feed Forward Network
        - Apply two fully connected layers with activation.
        - Process each token independently.
        - Apply Residual Connection and LayerNorm.

    Returns:
        decoder_output:
            Final decoder representation after masked self-attention,
            encoder-decoder attention, and feed-forward processing.

    Architecture:

        decoder_input
              ↓
      Masked Self-Attention
          (Q,K,V from Decoder)
              ↓
            Add & Norm
              ↓
      Encoder-Decoder Attention
      (Q from Decoder, K/V from Encoder)
              ↓
            Add & Norm
              ↓
       Feed Forward Network
              ↓
            Add & Norm
              ↓
         decoder_output
    """
    def forward(self,
                decoder_input,
                encoder_output,
                src_mask,
                tgt_mask):

        batch_size = decoder_input.shape[0]
        decoder_seq_len = decoder_input.shape[1]

        # ==========================================
        # STEP 1 : Masked Self Attention
        # ==========================================
        '''
            # Generate Query (Q), Key (K), and Value (V) vectors
            # from the decoder input using learned linear projections.
            #
            # Shape:
            # (batch_size, seq_len, d_model)
            #           ↓
            # (batch_size, seq_len, d_model)
            #
            # Then split each projection into multiple attention heads.
            #
            # Example:
            # (32, 10, 512)
            #        ↓
            # (32, 8, 10, 64)
            #
            # This allows each attention head to learn different
            # relationships and patterns between tokens.
        '''
        Q = self.self_Wq(decoder_input)
        K = self.self_Wk(decoder_input)
        V = self.self_Wv(decoder_input)

        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        # Compute masked self-attention.
        # Q, K, and V are generated from the decoder input.
        # tgt_mask prevents attention to future tokens and PAD tokens.
        '''
        Target sequence(tgt_mask):

        <START> I love PAD PAD

        Padding mask:

        1 1 1 0 0

        Look-ahead mask:

        1 0 0 0 0
        1 1 0 0 0
        1 1 1 0 0
        1 1 1 1 0
        1 1 1 1 1

        Combined:

        1 0 0 0 0
        1 1 0 0 0
        1 1 1 0 0
        0 0 0 0 0
        0 0 0 0 0
        '''
        attention_output = self.attention(
            Q,
            K,
            V,
            tgt_mask
        )

        # Combine all attention heads back into a single d_model vector.
        #
        # Shape transformation:
        # (batch_size, num_heads, seq_len, head_dim)
        #                ↓
        # (batch_size, seq_len, num_heads, head_dim)
        #                ↓
        # (batch_size, seq_len, d_model)
        #
        # Example:
        # (32, 8, 10, 64) → (32, 10, 512)

        # Multi-Head Attention produces separate outputs for each head.
        # Transpose moves the num_heads dimension back after seq_len.
        # Then all heads are concatenated together to reconstruct
        # the original d_model representation.
        #
        # Example:
        # Head1(64) + Head2(64) + ... + Head8(64)
        #                ↓
        #            512 features
        attention_output = (
            attention_output
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                decoder_seq_len,
                self.d_model
            )
        )

        attention_output = self.self_Wo(
            attention_output
        )

        attention_output = self.dropout1(
            attention_output
        )

        decoder_output = self.norm1(
            decoder_input + attention_output
        )

        # ==========================================
        # STEP 2 : Encoder Decoder Attention
        # ==========================================

        encoder_seq_len = encoder_output.shape[1]

        # Generate Query from decoder output
        Q = self.cross_Wq(
            decoder_output
        )

        # Generate Key and Value from encoder output
        K = self.cross_Wk(
            encoder_output
        )

        V = self.cross_Wv(
            encoder_output
        )

        # Split Query into multiple attention heads
        Q = self.split_heads(Q)

        # Split Key into multiple attention heads
        K = K.view(
            batch_size,
            encoder_seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # Split Value into multiple attention heads
        V = V.view(
            batch_size,
            encoder_seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        #Compute cross-attention using decoder queries
        # and encoder keys/values.
        cross_output = self.attention(
            Q,
            K,
            V,
            src_mask
        )

        # Concatenate all attention heads back into d_model
        cross_output = (
            cross_output
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                decoder_seq_len,
                self.d_model
            )
        )

        # Output projection
        cross_output = self.cross_Wo(
            cross_output
        )

        # Apply dropout
        cross_output = self.dropout2(
            cross_output
        )

        # Residual connection + Layer Normalization
        decoder_output = self.norm2(
            decoder_output + cross_output
        )

        # ==========================================
        # STEP 3 : Feed Forward Network
        # ==========================================

        ffn_output = self.ffn(
            decoder_output
        )

        ffn_output = self.dropout3(
            ffn_output
        )

        decoder_output = self.norm3(
            decoder_output + ffn_output
        )

        return decoder_output

In [16]:
import torch
import torch.nn as nn
import math

# TransformerEncoder (already in b8d8e772, moved here for order)
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ffn_hidden_dim, num_layers, max_seq_len, dropout_rate=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, ffn_hidden_dim, dropout_rate)
            for _ in range(num_layers)
        ])

    def forward(self, src, src_mask):
        x = self.embedding(src)
        x = self.positional_encoding(x)
        for layer in self.encoder_layers:
            x = layer(x, mask=src_mask)
        return x

# TransformerDecoder (from cell p55tJHues3fz)
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ffn_hidden_dim, num_layers, max_seq_len, dropout_rate=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, ffn_hidden_dim, dropout_rate)
            for _ in range(num_layers)
        ])

    def forward(self, tgt, enc_output, src_mask, tgt_mask):
        x = self.embedding(tgt)
        x = self.positional_encoding(x)
        for layer in self.decoder_layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return x

# FullTransformer (already in b8d8e772)
class FullTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, ffn_hidden_dim, num_encoder_layers, num_decoder_layers, max_seq_len, dropout_rate=0.1):
        super().__init__()
        self.encoder = TransformerEncoder(
            src_vocab_size, d_model, num_heads, ffn_hidden_dim, num_encoder_layers, max_seq_len, dropout_rate
        )
        self.decoder = TransformerDecoder(
            tgt_vocab_size, d_model, num_heads, ffn_hidden_dim, num_decoder_layers, max_seq_len, dropout_rate
        )
        self.output_linear = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask, tgt_mask):
        enc_output = self.encoder(src, src_mask)
        dec_output = self.decoder(tgt, enc_output, src_mask, tgt_mask)
        output = self.output_linear(dec_output)
        return output

# Model Parameters (re-using existing values from the notebook where appropriate)
src_vocab_size_full_transformer = vocab_size
tgt_vocab_size_full_transformer = vocab_size
num_layers = 2
num_heads = 2
ffn_hidden_dim = 32
num_encoder_layers_full_transformer = num_layers
num_decoder_layers_full_transformer = num_layers
dropout_rate_full_transformer = 0.1

# Instantiate the FullTransformer model
full_transformer_model = FullTransformer(
    src_vocab_size=src_vocab_size_full_transformer,
    tgt_vocab_size=tgt_vocab_size_full_transformer,
    d_model=d_model,
    num_heads=num_heads,
    ffn_hidden_dim=ffn_hidden_dim,
    num_encoder_layers=num_encoder_layers_full_transformer,
    num_decoder_layers=num_decoder_layers_full_transformer,
    max_seq_len=max_seq_len,
    dropout_rate=dropout_rate_full_transformer
)

print("FullTransformer instance created successfully!")
print(full_transformer_model)

FullTransformer instance created successfully!
FullTransformer(
  (encoder): TransformerEncoder(
    (embedding): Embedding(39, 8)
    (positional_encoding): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder_layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (Wq): Linear(in_features=8, out_features=8, bias=True)
        (Wk): Linear(in_features=8, out_features=8, bias=True)
        (Wv): Linear(in_features=8, out_features=8, bias=True)
        (Wo): Linear(in_features=8, out_features=8, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (layer_norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
        (ffn): Sequential(
          (0): Linear(in_features=8, out_features=32, bias=True)
          (1): ReLU()
          (2): Linear(in_features=32, out_features=8, bias=True)
        )
        (ffn_dropout): Dropout(p=0.1, inplace=False)
        (layer_norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=T